In [ ]:
#Qwen
import torch

import torch
def load_features(name,dims=10000):
    return torch.load(name)[:dims]

eng_features = load_features('activations/activation_engineering_base_qwen_st.pt')
che_features = load_features('activations/activation_chemistry_base_qwen_st.pt')
phy_features = load_features('activations/activation_physics_base_qwen_st.pt')
com_features = load_features('activations/activation_computer_base_qwen_st.pt')
oth_features = load_features('activations/activation_other_base_qwen_st.pt')
psy_features = load_features('activations/activation_psychology_base_qwen_st.pt')
his_features = load_features('activations/activation_history_base_qwen_st.pt')
law_features = load_features('activations/activation_law_base_qwen_st.pt')




limo_base_features = torch.load('activations/activation_limo_base_qwen_st.pt')[:10000]
limo_icl_features = torch.load('activations/activation_limo_icl_qwen_st.pt')[:10000]


import numpy
def calculate_minus_temp(features,sft_features):
    minus_eng = features - sft_features
    minus_eng = minus_eng**2
    minus_eng_sum = torch.sum(minus_eng,dim=0)
    return minus_eng_sum


def calculate_minus_temp2(features,sft_features,index=[i for i in range(65536)]):
    features = features - sft_features
    features = features**2
    minus_eng_sum = torch.mean(features,dim=0)
    features = torch.mean(features,dim=0)[index]
    return torch.mean(features)


#step 1: estimate shifted dimensions
index_limo_icl = torch.sort(calculate_minus_temp(limo_icl_features,limo_base_features),descending=True)[1][:100]


#step2 estimate correlations with downstream domians
eng = calculate_minus_temp2(eng_features,0,index_limo_icl)
che = calculate_minus_temp2(che_features,0,index_limo_icl)
phy = calculate_minus_temp2(phy_features,0,index_limo_icl)
com = calculate_minus_temp2(com_features,0,index_limo_icl)
other= calculate_minus_temp2(oth_features,0,index_limo_icl)
psy = calculate_minus_temp2(psy_features,0,index_limo_icl)
his = calculate_minus_temp2(his_features,0,index_limo_icl)
law = calculate_minus_temp2(law_features,0,index_limo_icl)




x=[eng.item(),che.item(),phy.item(),com.item(),other.item(),psy.item(),his.item(),law.item()]

#real performance change
y=[0.0980, 0.0636, 0.0524, 0.0488, 0.0130, 0.0037, 0.0026, 0.0019]

print(x,y)

import numpy as np
correlation_matrix = np.corrcoef(x, y)
correlation_coefficient = correlation_matrix[0, 1]

print(f"coefficient: {correlation_coefficient}")
